In [1]:
import numpy as np
# ----- JAX IN ------------------
from typing import List
import haiku as hk
import jax
import InteractionNet
from InteractionNet.jax import deep_typed_graph_net
from InteractionNet.jax import typed_graph
from InteractionNet.jax import model_utils 
from InteractionNet.jax import grid_mesh_connectivity  # Assuming you have the connectivity utility

## InteractionNet JAX
---

In [2]:
def init_grid2mesh_graph(
    grid_lat: np.ndarray,
    grid_lon: np.ndarray,
    finest_mesh: List,
    query_radius: float,
    grid_nodes_lat: np.ndarray,
    grid_nodes_lon: np.ndarray,
    mesh_nodes_lat: np.ndarray,
    mesh_nodes_lon: np.ndarray,
    num_grid_nodes: int,
    num_mesh_nodes: int,
    spatial_features_kwargs: dict,
) -> typed_graph.TypedGraph:
    """Build Grid2Mesh graph."""
    
    # Create some edges according to distance between mesh and grid nodes.
    assert grid_lat is not None and grid_lon is not None
    grid_indices, mesh_indices = grid_mesh_connectivity.radius_query_indices(
        grid_latitude=grid_lat,
        grid_longitude=grid_lon,
        mesh=finest_mesh,
        radius=query_radius,
    )

    # Edges sending info from grid to mesh.
    senders = grid_indices
    receivers = mesh_indices

    # Precompute structural node and edge features according to config options.
    (senders_node_features, receivers_node_features, edge_features) = model_utils.get_bipartite_graph_spatial_features(
        senders_node_lat=grid_nodes_lat,
        senders_node_lon=grid_nodes_lon,
        receivers_node_lat=mesh_nodes_lat,
        receivers_node_lon=mesh_nodes_lon,
        senders=senders,
        receivers=receivers,
        edge_normalization_factor=None,
        **spatial_features_kwargs,
    )

    n_grid_node = np.array([num_grid_nodes])
    n_mesh_node = np.array([num_mesh_nodes])
    n_edge = np.array([mesh_indices.shape[0]])

    # Create NodeSet for grid and mesh nodes
    grid_node_set = typed_graph.NodeSet(
        n_node=n_grid_node, features=senders_node_features
    )
    mesh_node_set = typed_graph.NodeSet(
        n_node=n_mesh_node, features=receivers_node_features
    )

    # Create EdgeSet
    edge_set = typed_graph.EdgeSet(
        n_edge=n_edge,
        indices=typed_graph.EdgesIndices(senders=senders, receivers=receivers),
        features=edge_features,
    )

    # Create Node and Edge Dictionaries
    nodes = {"grid_nodes": grid_node_set, "mesh_nodes": mesh_node_set}
    edges = {
        typed_graph.EdgeSetKey("grid2mesh", ("grid_nodes", "mesh_nodes")): edge_set
    }

    # Create the TypedGraph object
    grid2mesh_graph = typed_graph.TypedGraph(
        context=typed_graph.Context(n_graph=np.array([1]), features=()),
        nodes=nodes,
        edges=edges,
    )

    return grid2mesh_graph

In [3]:
finest_mesh = InteractionNet.jax.finest_mesh
mesh_nodes_lat, mesh_nodes_lon = (
    InteractionNet.jax.mesh_nodes_lat, 
    InteractionNet.jax.mesh_nodes_lon,
    )  # Example mesh nodes latitude and longitude
num_mesh_nodes = finest_mesh.vertices.shape[0]  # Number of mesh nodes
# -----------------------------------------------------
# Init grid properties
# Sample data (replace with your actual data)
grid_lat = np.linspace(-90, 90, 300).astype(np.float32)  # Example grid latitude
grid_lon = np.linspace(-180, 180, 300).astype(np.float32)   # Example grid longitude
num_grid_nodes = grid_lat.shape[0] * grid_lon.shape[0] # Number of grid nodes
grid_nodes_lon, grid_nodes_lat = np.meshgrid(grid_lon, grid_lat)
grid_nodes_lat = grid_nodes_lat.reshape([-1]).astype(np.float32) # Example grid nodes latitude 
grid_nodes_lon = grid_nodes_lon.reshape([-1]).astype(np.float32)   # Example grid nodes longitude
# -----------------------------------------------------
spatial_features_kwargs = dict(
        add_node_positions=False,
        add_node_latitude=True,
        add_node_longitude=True,
        add_relative_positions=True,
        relative_longitude_local_coordinates=True,
        relative_latitude_local_coordinates=True,
    )  # Additional feature computation args (if any)
query_radius = 0.5  # Example query radius

# Create the graph
input_graph = init_grid2mesh_graph(
    grid_lat, grid_lon, finest_mesh, query_radius,
    grid_nodes_lat, grid_nodes_lon, mesh_nodes_lat, mesh_nodes_lon,
    num_grid_nodes, num_mesh_nodes, spatial_features_kwargs
)

In [4]:
input_graph

TypedGraph(context=Context(n_graph=array([1]), features=()), nodes={'grid_nodes': NodeSet(n_node=array([90000]), features=array([[-1.0000000e+00, -1.0000000e+00,  8.7422777e-08],
       [-1.0000000e+00, -9.9977922e-01, -2.1012342e-02],
       [-1.0000000e+00, -9.9911696e-01, -4.2015493e-02],
       ...,
       [ 1.0000000e+00, -9.9911696e-01,  4.2015493e-02],
       [ 1.0000000e+00, -9.9977922e-01,  2.1012342e-02],
       [ 1.0000000e+00, -1.0000000e+00, -8.7422777e-08]],
      shape=(90000, 3), dtype=float32)), 'mesh_nodes': NodeSet(n_node=array([642]), features=array([[ 0.18759258,  0.49999997,  0.86602545],
       [ 0.7946544 , -0.50000006,  0.8660254 ],
       [ 0.7946544 ,  1.        ,  0.        ],
       ...,
       [ 0.34392706, -0.7251567 ,  0.6885838 ],
       [ 0.24886788, -0.8127705 ,  0.58258396],
       [ 0.40267685, -0.8279131 ,  0.5608564 ]],
      shape=(642, 3), dtype=float32))}, edges={EdgeSetKey(name='grid2mesh', node_sets=('grid_nodes', 'mesh_nodes')): EdgeSet(n_ed

In [5]:
def GraphCastIN(input_graph: typed_graph.TypedGraph):
    # Create the model object and call it to perform a forward pass
    GNN = deep_typed_graph_net.DeepTypedGraphNet(
        embed_nodes=True,  # Embed raw features of the grid and mesh nodes.
        embed_edges=True,  # Embed raw features of the grid2mesh edges.
        edge_latent_size=dict(
            grid2mesh=InteractionNet.jax.model_config_GC.latent_size
            ),
        node_latent_size=dict(
            mesh_nodes=InteractionNet.jax.model_config_GC.latent_size,
            grid_nodes=InteractionNet.jax.model_config_GC.latent_size
            ),
        mlp_hidden_size=InteractionNet.jax.model_config_GC.latent_size,
        mlp_num_hidden_layers=InteractionNet.jax.model_config_GC.hidden_layers,
        num_message_passing_steps=1,
        use_layer_norm=True,
        include_sent_messages_in_node_update=False,
        activation="swish",
        f32_aggregation=True,
        aggregate_normalization=None,
        name="grid2mesh_gnn",
    )
    return GNN(input_graph)

# Initialize the Haiku module with hk.transform
transformed_model = hk.transform(GraphCastIN)

# Initialize a random number generator (rng) for parameter initialization
rng = jax.random.PRNGKey(42)

# Initialize parameters using Haiku
params = transformed_model.init(rng, input_graph)

# Now, run a forward pass using the initialized parameters
output = transformed_model.apply(params, rng, input_graph)

In [6]:
output

TypedGraph(context=Context(n_graph=array([1]), features=()), nodes={'grid_nodes': NodeSet(n_node=array([90000]), features=Array([[ 1.2147334 ,  0.09915441, -1.4598231 , ..., -0.08649278,
         0.00716498,  0.81601095],
       [ 1.1587334 ,  0.12516439, -1.4319564 , ..., -0.1093992 ,
         0.01656108,  0.78609705],
       [ 1.1038471 ,  0.15086055, -1.4047258 , ..., -0.13230228,
         0.02543581,  0.7561774 ],
       ...,
       [ 0.6683941 ,  2.2378016 , -0.49099684, ..., -1.1140087 ,
         1.5430377 , -2.5032372 ],
       [ 0.61349076,  2.248906  , -0.48846525, ..., -1.1173705 ,
         1.5374691 , -2.5480797 ],
       [ 0.5578298 ,  2.2586937 , -0.48531955, ..., -1.1202302 ,
         1.5315908 , -2.5927405 ]], dtype=float32)), 'mesh_nodes': NodeSet(n_node=array([642]), features=Array([[-1.3439456 ,  0.594499  , -1.1344244 , ..., -0.8283068 ,
        -0.32669768,  0.8738307 ],
       [ 0.00599247,  0.7957479 , -2.6213849 , ...,  0.61909974,
        -0.04922104,  0.7640488